# 06 · Paper tables and figures

Builds every table (`paper_mdpi_fintech/tables/*.tex`) and figure (`paper_mdpi_fintech/figures/*.pdf`) used in the manuscript **directly from the result files** written by notebooks 02–05, so no number in the paper is typed by hand.

Run it after notebooks 01–05 whenever results change, then recompile `paper_mdpi_fintech/main.tex` (e.g., on Overleaf).

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "src" / "exgap").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

## Setup

In [ ]:
import shutil
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats as sps
from exgap.build_panel import load_panel
from exgap import stats as st
from exgap.config import ROOT, TABLES_DIR, FIGURES_DIR

PAPER = ROOT / "paper_mdpi_fintech"
PT, PF = PAPER / "tables", PAPER / "figures"
PT.mkdir(parents=True, exist_ok=True); PF.mkdir(parents=True, exist_ok=True)
T = lambda name: pd.read_csv(TABLES_DIR / f"{name}.csv")
st.set_style()
plt.rcParams.update({"axes.titlesize": 10, "font.size": 9})
SHORT = {"521CI": "Banking (521CI)", "523": "Securities (523)", "524": "Insurance (524)"}
SOURCE = "Source: authors' calculations from BEA, BLS, FRED and Census data (vintage of 22 November 2026)."


def tex_escape(s):
    return str(s).replace("&", r"\&").replace("%", r"\%").replace("_", r"\_").replace("#", r"\#")


def stars_tex(s):
    s = str(s)
    for k in ["***", "**", "*"]:
        if s.endswith(k):
            return s[: -len(k)] + "$^{" + k + "}$"
    return s


def num(x, d=3):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "--"
    return f"{x:.{d}f}".replace("-", "$-$")


def pval(p):
    if p is None or np.isnan(p):
        return "--"
    return "$<$0.001" if p < 0.001 else f"{p:.3f}"


def star(p):
    return "$^{***}$" if p < 0.01 else "$^{**}$" if p < 0.05 else "$^{*}$" if p < 0.10 else ""


def write_table(name, caption, label, colspec, header, rows, note=None, wide=False, size=None):
    lines = []
    if wide:
        lines.append(r"\begin{table}[H]")
        lines.append(r"\begin{adjustwidth}{-\extralength}{0cm}")
    else:
        lines.append(r"\begin{table}[H]")
    lines.append(r"\caption{" + caption + "}")
    lines.append(r"\label{" + label + "}")
    if size:
        lines.append("\\" + size)
    lines.append(r"\begin{tabular}{" + colspec + "}")
    lines.append(r"\toprule")
    lines += header if isinstance(header, list) else [header]
    lines.append(r"\midrule")
    lines += rows
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    if note:
        lines.append(r"\noindent{\footnotesize{" + note + "}}")
    if wide:
        lines.append(r"\end{adjustwidth}")
    lines.append(r"\end{table}")
    (PT / f"{name}.tex").write_text("% ---- table: " + name + " (generated by 06_paper_outputs.ipynb; do not edit by hand) ----\n"
                                    + "\n".join(lines) + "\n", encoding="utf-8")
    print("wrote tables/" + name + ".tex")

df = load_panel()
print(PAPER, "|", df.code.nunique(), "industries")

## Table data sources

In [ ]:
rows = [
    r"Value added (nominal, real) & BEA GDP by Industry, Tables 1 and 10 & 1997--2025 \\",
    r"Compensation of employees; gross operating surplus & BEA GDP by Industry, Table 6 & 1997--2025 \\",
    r"Full-time-equivalent employees & BEA NIPA Table 6.5D & 1998--2025 \\",
    r"IPP net stock (current cost; chain-type quantity index) & BEA Fixed Assets, Tables 3.1I and 3.2I & 1997--2025 \\",
    r"IPP investment & BEA Fixed Assets, Table 3.7I & 1997--2025 \\",
    r"Equipment and total private fixed-asset stocks & BEA Fixed Assets, Tables 3.1E and 3.1ESI & 1997--2025 \\",
    r"Interest rates, credit spread, VIX, NASDAQ, PCE prices, e-commerce share, recessions & FRED & 1997--2025 \\",
    r"Finance sub-sector employment; nonfarm labor share and productivity & BLS CES and Productivity programs & 1997--2025 \\",
    r"Business applications by sector & Census Business Formation Statistics & 2005--2025 \\",
]
write_table("tab_data_sources", "Data sources. All series are public and were retrieved through the providers' APIs; "
            "BEA tables were located by their published descriptions at run time. " + SOURCE,
            "tab:datasources", "p{6.2cm}p{5.2cm}l", r"Variable & Source & Coverage \\", rows, wide=True, size="small")

## Table descriptives

In [ ]:
d1 = T("table1_descriptives").rename(columns={"Unnamed: 0": "var"})
keep = [("Labour share", "Labor share", 1, 3), ("log real IPP capital per worker (index)", "log real IPP per worker (index)", 1, 2),
        ("IPP stock / value added", "IPP stock / value added", 1, 3), ("IPP share of fixed assets", "IPP share of fixed assets", 1, 3),
        ("IPP stock growth (log)", "IPP stock growth (log, nominal)", 1, 3),
        ("Real VA per FTE (2017 $)", "Real value added per FTE (2017 \\$000)", 1e-3, 1),
        ("Real pay per FTE (2017 $)", "Real pay per FTE (2017 \\$000)", 1e-3, 1),
        ("Productivity-pay gap (log pts)", "Productivity--pay gap (log points)", 1, 3)]
rows = []
for raw, label, scale, dd in keep:
    r = d1[d1["var"] == raw].set_index("sample")
    if r.empty:
        continue
    cells = []
    for smp in ["Finance", "Non-finance"]:
        cells += [num(r.loc[smp, "mean"] * scale, dd), num(r.loc[smp, "std"] * scale, dd)]
    rows.append(label + " & " + " & ".join(cells) + r" \\")
nf = int(d1[(d1["var"] == "Labour share") & (d1["sample"] == "Finance")]["count"].iloc[0])
nn = int(d1[(d1["var"] == "Labour share") & (d1["sample"] == "Non-finance")]["count"].iloc[0])
write_table("tab_descriptives", f"Descriptive statistics, industry-years 1997--2025. Finance comprises banking (521CI), securities "
            f"(523) and insurance (524), {nf} industry-years; non-finance comprises 61 private industries, {nn} industry-years. "
            "Per-worker and productivity variables start in 1998, when BEA's employment table begins. " + SOURCE,
            "tab:descriptives", "lcccc",
            [r" & \multicolumn{2}{c}{Finance} & \multicolumn{2}{c}{Non-finance} \\", r"\cmidrule(lr){2-3}\cmidrule(lr){4-5}",
             r"Variable & Mean & SD & Mean & SD \\"], rows)

## Table H1

In [ ]:
a = T("h1a_exponential_trends").rename(columns={"Unnamed: 0": "series"})
lab = {"521CI": "Banking (521CI)", "523": "Securities (523)", "524": "Insurance (524)",
       "ALL FINANCE": "Finance, value-added weighted", "ALL NON-FINANCE": "Non-finance, value-added weighted"}
rows = [r"\multicolumn{5}{l}{\textit{Panel A. Log-linear trend in real IPP net stock, 1997--2025}} \\"]
for _, r in a.iterrows():
    key = next(k for k in lab if str(r.series).startswith(k))
    rows.append(f"{lab[key]} & {100*r.growth_rate:.2f}\\% & {num(r.se, 4)} & {r.doubling_years:.1f} & "
                f"{r.rmse_exponential:.1f} / {r.rmse_linear:.1f}" + r" \\")
b = T("h1b_growth_premium").set_index("Unnamed: 0")
c = T("h1c_structural_breaks").set_index("Unnamed: 0")
rows += [r"\midrule", r"\multicolumn{5}{l}{\textit{Panel B. Annual IPP stock growth, panel with year effects (SE clustered by industry)}} \\",
         f"Finance growth premium & {stars_tex(b.loc['finance', '(1) Finance premium'])} & {b.loc['finance (se)', '(1) Finance premium']} & & " + r"\\",
         f"Finance $\\times$ post-2010 (industry FE) & {stars_tex(b.loc['fin_x_post', '(3) Industry FE'])} & {b.loc['fin_x_post (se)', '(3) Industry FE']} & & " + r"\\",
         r"\midrule", r"\multicolumn{5}{l}{\textit{Panel C. Break tests, finance aggregate}} \\",
         f"Growth difference after 2010 (Newey--West) & {num(c.loc['Growth difference (HAC)', 'statistic'], 3)} & $p$ = {pval(c.loc['Growth difference (HAC)', 'p_value'])} & & " + r"\\",
         f"Quandt--Andrews sup-$F$ (break in {c.loc['Quandt-Andrews sup-F', 'detail'].split('year ')[1].split(';')[0]}) & {c.loc['Quandt-Andrews sup-F', 'statistic']:.1f} & $p_{{boot}}$ = {pval(c.loc['Quandt-Andrews sup-F', 'p_value'])} & & " + r"\\",
         f"Chow test at 2010 (independent errors) & {c.loc['Chow test', 'statistic']:.1f} & $p$ = {pval(c.loc['Chow test', 'p_value'])} & & " + r"\\",
         f"PELT change points in growth & none & & & " + r"\\"]
write_table("tab_h1_growth", "IPP capital growth in finance (H1). Panel A: annual growth implied by a log-linear trend "
            "with Newey--West standard errors; the last column compares the root-mean-square error of exponential and linear "
            "trends fitted to the level. Panel B: nominal (current-cost) IPP stock growth. Panel C: the sup-$F$ $p$-value uses a "
            "moving-block bootstrap. " + SOURCE, "tab:h1", "lcccc",
            r"Series / test & Estimate & SE / $p$ & Doubling (yrs) & RMSE exp./lin. \\", rows, wide=True, size="small")

## Table H3 main

In [ ]:
h = T("h3_labor_share_fe").set_index("Unnamed: 0")
names = {"L_ipp": r"$\log(\text{IPP}/\text{FTE})_{t-1}$", "fin_x_L_ipp": r"\quad $\times$ Finance",
         "post_x_L_ipp": r"\quad $\times$ Post-2010", "fin_post_x_L_ipp": r"\quad $\times$ Finance $\times$ Post-2010",
         "L_eq": r"$\log(\text{Equipment}/\text{FTE})_{t-1}$"}
rows = []
for term, label in names.items():
    rows.append(label + " & " + " & ".join(stars_tex(v) if isinstance(v, str) else "" for v in h.loc[term].fillna("")) + r" \\")
    rows.append(" & " + " & ".join(v if isinstance(v, str) else "" for v in h.loc[term + " (se)"].fillna("")) + r" \\")
rows.append(r"\midrule")
for lab_, key in [("Observations", "Observations"), ("Industries", "Industries"), ("$R^2$ net of fixed effects", "R2 (net of FE)")]:
    rows.append(lab_ + " & " + " & ".join(h.loc[key].astype(str)) + r" \\")
rows.append(r"Standard errors & Ind. & Ind. & Ind. & Ind. & D--K & Two-way \\")
write_table("tab_h3_fe", "IPP capital deepening and the labor share: two-way fixed-effects estimates (H3). Dependent "
            "variable: compensation of employees divided by value added. All models include industry and year fixed effects. "
            "Standard errors in parentheses: clustered by industry (Ind.), Driscoll--Kraay (D--K), or clustered by industry and "
            "year (Two-way). $^{*}$ $p<0.10$, $^{**}$ $p<0.05$, $^{***}$ $p<0.01$. " + SOURCE,
            "tab:h3fe", "lcccccc", r" & (1) & (2) & (3) & (4) & (5) & (6) \\", rows, wide=True, size="small")

## Table inference summary

In [ ]:
me = T("h3_marginal_effects").set_index("Unnamed: 0")
ri = T("h3_randomization_inference").iloc[0]
fd = T("h3_first_differences").set_index("Unnamed: 0")
dml = T("h3_dml").iloc[0]
lead = T("robust_lead_test").set_index("term")
va = T("h3_secondary_ipp_per_va").set_index("Unnamed: 0")
def p_from(coef_str, se_str):
    b_ = float(coef_str.rstrip("*")); s_ = float(se_str.strip("()"))
    return b_, s_, 2 * (1 - sps.norm.cdf(abs(b_ / s_)))
fd_b, fd_s, fd_p = p_from(fd.loc["d_fin_x_L_ipp", "(2) FD x Finance"], fd.loc["d_fin_x_L_ipp (se)", "(2) FD x Finance"])
va_b, va_s, va_p = p_from(va.loc["L_log_ipp_intensity", "(1) IPP/VA"], va.loc["L_log_ipp_intensity (se)", "(1) IPP/VA"])
vf_b, vf_s, vf_p = p_from(va.loc["fin_x_L_ipp_va", "(2) IPP/VA x Finance"], va.loc["fin_x_L_ipp_va (se)", "(2) IPP/VA x Finance"])
h3b, h3s, h3p = p_from(h.loc["fin_x_L_ipp", "(2) x Finance"], h.loc["fin_x_L_ipp (se)", "(2) x Finance"])
rows = [r"\multicolumn{4}{l}{\textit{Panel A. Effects implied by model (2) of Table~\ref{tab:h3fe}}} \\",
        f"Non-finance effect, $\\beta_1$ & {num(me.loc['Non-finance', 'effect'])} & {num(me.loc['Non-finance', 'se'])} & {pval(me.loc['Non-finance', 'p'])}" + r" \\",
        f"Finance net effect, $\\beta_1+\\beta_2$ & {num(me.loc['Finance', 'effect'])} & {num(me.loc['Finance', 'se'])} & {pval(me.loc['Finance', 'p'])}" + r" \\",
        f"Finance difference, $\\beta_2$ (industry-clustered) & {num(h3b)} & {num(h3s)} & {pval(h3p)}" + r" \\",
        f"Finance difference, $\\beta_2$ (randomization inference, one-sided) & {num(ri.actual_interaction)} & -- & {pval(ri.one_sided_p_randomization)}" + r" \\",
        r"\midrule", r"\multicolumn{4}{l}{\textit{Panel B. Alternative estimators}} \\",
        f"First differences: finance difference & {num(fd_b)} & {num(fd_s)} & {pval(fd_p)}" + r" \\",
        f"Double machine learning: average effect, all industries & {num(dml.theta)} & {num(dml.se)} & {pval(dml.p)}" + r" \\",
        r"\midrule", r"\multicolumn{4}{l}{\textit{Panel C. Specification checks}} \\",
        f"Lead test: $\\log(\\text{{IPP}}/\\text{{FTE}})_{{t+2}}$ & {num(lead.loc['LEAD2', 'coef'])} & {num(lead.loc['LEAD2', 'se'])} & {pval(lead.loc['LEAD2', 'p'])}" + r" \\",
        f"IPP/value added (division bias): all industries & {num(va_b)} & {num(va_s)} & {pval(va_p)}" + r" \\",
        f"IPP/value added (division bias): finance difference & {num(vf_b)} & {num(vf_s)} & {pval(vf_p)}" + r" \\"]
write_table("tab_h3_inference", "Inference for the finance effect and alternative estimators. Randomization inference reassigns the "
            "finance label to 1{,}000 random groups of three industries. Double machine learning uses random-forest nuisance "
            "models, five folds and three repetitions, with standard errors clustered by industry. The lead test adds IPP capital "
            "per worker two years ahead to the baseline model. The IPP/value-added rows use the treatment that shares a "
            "denominator with the labor share. " + SOURCE, "tab:h3inf", "lccc", r"Estimate & Coefficient & SE & $p$-value \\", rows,
            wide=True, size="small")

## Table robustness

In [ ]:
rb = T("robustness_summary").set_index("check")
order = [("Baseline", "Baseline"), ("Measure: nominal IPP stock per worker", "Measure: nominal IPP stock per worker"),
         ("Measure: IPP investment per worker", "Measure: IPP investment per worker"),
         ("Measure: IPP share of fixed assets", "Measure: IPP share of fixed assets$^{a}$"),
         ("Measure: IPP / value added (division bias)", "Measure: IPP / value added (division bias)"),
         ("Sample: excl. COVID years", "Sample: excluding 2020--2021"),
         ("Sample: excl. IT-producing industries", "Sample: excluding IT-producing industries"),
         ("Sample: incl. funds & trusts (525)", "Sample: including funds and trusts (525)"),
         ("Dynamics: two-year lag", "Dynamics: two-year lag"), ("Dynamics: 5-year long differences", "Dynamics: five-year long differences"),
         ("Weighting: value-added weights", "Weighting: value-added weights"),
         ("Outcome: log real pay per FTE", "Outcome: log real pay per FTE$^{b}$")]
rows = []
for key, label in order:
    if key not in rb.index:
        continue
    r = rb.loc[key]
    pf = 2 * (1 - sps.norm.cdf(abs(r.beta_finance / r.se_finance))) if r.se_finance > 0 else np.nan
    rows.append(f"{tex_escape(label) if '$' not in label else label} & {num(r.beta_nonfinance)}{star(r.p_nonfinance)} & ({num(r.se_nonfinance)}) & "
                f"{num(r.beta_finance)}{star(pf)} & ({num(r.se_finance)}) & {int(r.nobs)}" + r" \\")
rows.append(r"\midrule")
rows.append(f"Randomization inference for finance difference, one-sided $p$ & \\multicolumn{{5}}{{l}}{{{pval(rb.loc['Randomization inference: finance label (one-sided p)', 'p_nonfinance'])}}}" + r" \\")
write_table("tab_robustness", "Robustness of the labor-share effect of IPP capital deepening. Each row re-estimates model (2) of "
            "Table~\\ref{tab:h3fe}; the finance effect is $\\beta_1+\\beta_2$. Standard errors clustered by industry. "
            "$^{a}$ Share of capital, not a logarithm, so coefficients are not comparable in magnitude. $^{b}$ Dependent variable is log "
            "real compensation per FTE rather than the labor share. $^{*}$ $p<0.10$, $^{**}$ $p<0.05$, $^{***}$ $p<0.01$. " + SOURCE,
            "tab:robust", "lccccc", r"Check & Non-finance & SE & Finance & SE & $N$ \\", rows, wide=True, size="small")

## Table decomposition

In [ ]:
dc = T("h2_gap_decomposition_finance").set_index("year")
yrs = [y for y in [1998, 2002, 2006, 2010, 2014, 2018, 2022, 2025] if y in dc.index]
rows = [f"{y} & {num(dc.loc[y, 'gap'])} & {num(dc.loc[y, 'from_labour_share'])} & {num(dc.loc[y, 'from_price_wedge'])}" + r" \\" for y in yrs]
h2 = T("h2_prod_pay_gap").set_index("Unnamed: 0")
write_table("tab_h2_decomp", "Decomposition of the productivity--pay gap in finance (value-added weighted, log points since 1998). "
            "The gap equals the cumulative fall in the labor share plus the cumulative rise of consumer prices (PCE) relative to "
            "finance output prices; the identity holds exactly in every year. " + SOURCE, "tab:h2decomp", "lccc",
            [r" & Productivity--pay & \multicolumn{2}{c}{Contribution from} \\", r"\cmidrule(lr){3-4}",
             r"Year & gap & labor-share decline & price wedge \\"], rows)

## Table causal forest

In [ ]:
ci = T("cf_industry_effects")
ate = T("cf_average_effect").iloc[0]
rk = T("cf_finance_rank_test").set_index("Unnamed: 0")["value"]
DAGGER = r" $\dagger$"  # outside the f-string: backslashes in f-string expressions need Python 3.12+
rows = [f"{int(r['rank'])} & {tex_escape(r['name'])} ({r['code']}){DAGGER if r.finance == 1 else ''} & {num(r.cate)} & [{num(r.ci_low)}, {num(r.ci_high)}]" + r" \\"
        for _, r in ci.head(10).iterrows()]
rows += [r"\midrule",
         f"\\multicolumn{{2}}{{l}}{{Average effect, all industries}} & {num(ate.point_estimate)} & [{num(ate.ci_lower)}, {num(ate.ci_upper)}]" + r" \\",
         f"\\multicolumn{{2}}{{l}}{{Finance mean effect (mean rank {rk['finance_mean_rank']:.1f} of {int(rk['n_industries'])})}} & {num(rk['finance_mean_effect'])} & " + r"\\",
         f"\\multicolumn{{2}}{{l}}{{Non-finance mean effect}} & {num(rk['other_mean_effect'])} & " + r"\\",
         f"\\multicolumn{{2}}{{l}}{{Rank test: finance vs 10{{,}}000 random groups of three, one-sided $p$}} & {pval(rk['one_sided_p_rank_test'])} & " + r"\\"]
write_table("tab_cf", "Causal forest: the ten industries with the most negative estimated effect of IPP capital per worker on the "
            "labor share, 2003--2025. The forest learns heterogeneity from industries' 1998--2002 characteristics and is not given "
            "the finance classification; $\\dagger$ marks finance industries. 95\\% confidence intervals from the forest's "
            "bootstrap-of-little-bags inference. " + SOURCE, "tab:cf", "clcc",
            r"Rank & Industry (BEA code) & Effect & 95\% CI \\", rows, wide=True, size="small")

## Table forecasting

In [ ]:
f1, f3, f3f = T("h4_oos_h1").set_index("Unnamed: 0"), T("h4_oos_h3").set_index("Unnamed: 0"), T("h4_oos_h3_finance").set_index("Unnamed: 0")
mnames = {"ZeroChange": "No change (benchmark)", "HistMean": "Historical mean", "Ridge": "Ridge", "Huber": "Huber regression",
          "RandomForest": "Random forest", "HistGB_L1": "Gradient boosting (L1 loss)", "LightGBM_Huber": "LightGBM (Huber loss)"}
rows = []
for m, lab_ in mnames.items():
    cells = []
    for tab in [f1, f3, f3f]:
        cells += [num(tab.loc[m, "MAE"], 4), "--" if m == "ZeroChange" else num(tab.loc[m, "DM_vs_ZeroChange_MAE"], 2),
                  "--" if m == "ZeroChange" else pval(tab.loc[m, "p_DM"])]
    rows.append(lab_ + " & " + " & ".join(cells) + r" \\")
cl = T("h4_classification_h3").set_index("Unnamed: 0")
rows += [r"\midrule", f"\\multicolumn{{10}}{{l}}{{Classification of three-year declines (AUC): logit {cl.loc['Logit', 'AUC']:.3f}; "
         f"random forest {cl.loc['RandomForest', 'AUC']:.3f}; base rate {cl.loc['BaseRate', 'AUC']:.3f}}}" + r" \\"]
write_table("tab_forecast", "Out-of-sample forecasts of labor-share changes (H4), expanding window with forecast origins 2008 "
            "onwards. MAE is the mean absolute error; DM is the Diebold--Mariano statistic for absolute-error loss against the "
            "no-change forecast (positive values favor the model), with the Harvey--Leybourne--Newbold correction. " + SOURCE,
            "tab:forecast", "lccccccccc",
            [r" & \multicolumn{3}{c}{All, 1 year} & \multicolumn{3}{c}{All, 3 years} & \multicolumn{3}{c}{Finance, 3 years} \\",
             r"\cmidrule(lr){2-4}\cmidrule(lr){5-7}\cmidrule(lr){8-10}",
             r"Model & MAE & DM & $p$ & MAE & DM & $p$ & MAE & DM & $p$ \\"], rows, wide=True, size="footnotesize")

## Table hypotheses

In [ ]:
hs = T("hypothesis_summary")
labels = {"H1a Finance IPP stock grows (trend growth > 0)": "H1a: finance IPP stock grows exponentially",
          "H1b Finance IPP growth premium": "H1b: faster IPP growth in finance",
          "H1b Extra post-2010 acceleration in finance": "H1b: extra post-2010 acceleration in finance",
          "H1c Finance IPP growth higher after 2010 (HAC)": "H1c: higher finance IPP growth after 2010",
          "H2 IPP intensity widens the productivity-pay gap (all)": "H2: IPP widens productivity--pay gap (all)",
          "H2 ... in finance (beta + beta_fin)": "H2: IPP widens productivity--pay gap (finance)",
          "H3 IPP intensity lowers the labour share (all)": "H3: IPP lowers labor share (all industries)",
          "H3 Net effect in finance (beta + beta_fin)": "H3: IPP lowers labor share (finance)",
          "H3 Effect more negative in finance (beta_fin, randomization p)": "H3: effect more negative in finance (randomization)",
          "H3 DML effect, all industries": "H3: IPP lowers labor share (DML, all industries)"}
rows = [f"{labels.get(r.hypothesis, tex_escape(r.hypothesis))} & {r.expected_sign} & {num(r.estimate, 4)} & {pval(r.p_one_sided)} & "
        f"{pval(r.p_holm)} & {'Yes' if r.supported_5pct_holm else 'No'}" + r" \\" for _, r in hs.iterrows()]
write_table("tab_hypotheses", "Summary of hypothesis tests. Each test is one-sided in the expected direction; an estimate with "
            "the wrong sign receives a $p$-value above 0.5. $p$-values are adjusted for multiple testing with the "
            "Holm--Bonferroni procedure across all ten tests. " + SOURCE, "tab:hypotheses", "lccccc",
            r"Hypothesis & Sign & Estimate & $p$ (one-sided) & $p$ (Holm) & Supported \\", rows, wide=True, size="small")

## Figure labor share

In [ ]:
m = df.copy()
m["group"] = np.where(m.finance == 1, "Finance (banking, securities, insurance)", "Other private industries")
agg = m.groupby(["year", "group"]).apply(lambda g: g["comp"].sum() / g["va_nom"].sum()).unstack()
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(agg.index, agg.iloc[:, 0], color=st.PALETTE["finance"], lw=2, label=agg.columns[0])
ax.plot(agg.index, agg.iloc[:, 1], color=st.PALETTE["other"], lw=2, label=agg.columns[1])
ax.axvline(2010, ls="--", color="k", lw=0.7, alpha=0.6)
ax.set_ylabel("Compensation / value added"); ax.legend(loc="lower left")
fig.savefig(PF / "fig_labor_share.pdf", bbox_inches="tight"); plt.show()

## Figure ipp per worker

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.2))
for code, lab_ in SHORT.items():
    s = df[df.code == code].dropna(subset=["log_ipp_pw"]).set_index("year")["log_ipp_pw"]
    ax.plot(s.index, 100 * np.exp(s - s.iloc[0]), lw=2, label=lab_)
nf_ = df[df.finance == 0].dropna(subset=["log_ipp_pw"]).copy()
nf_["idx"] = nf_["log_ipp_pw"] - nf_.groupby("code")["log_ipp_pw"].transform("first")
med = nf_.groupby("year")["idx"].median()
ax.plot(med.index, 100 * np.exp(med), color=st.PALETTE["other"], lw=2, ls="--", label="Non-finance median")
ax.set_yscale("log"); ax.set_ylabel("Real IPP capital per worker\n(1998 = 100, log scale)"); ax.legend(fontsize=8)
fig.savefig(PF / "fig_ipp_per_worker.pdf", bbox_inches="tight"); plt.show()

## Figure decomposition

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(dc.index, dc["gap"], color="k", lw=2, label="Productivity--pay gap")
ax.plot(dc.index, dc["from_labour_share"], color=st.PALETTE["finance"], lw=2, label="From labor-share decline")
ax.plot(dc.index, dc["from_price_wedge"], color=st.PALETTE["information"], lw=2, ls="--", label="From consumer/output price wedge")
ax.axhline(0, color="k", lw=0.5); ax.set_ylabel("Log points since 1998"); ax.legend(fontsize=8)
fig.savefig(PF / "fig_gap_decomposition.pdf", bbox_inches="tight"); plt.show()

## Figure robustness

In [ ]:
plot_rows = [k for k, _ in order if k in rb.index and not k.startswith(("Measure: IPP share", "Outcome"))]
pr = rb.loc[plot_rows]
fig, ax = plt.subplots(figsize=(6.5, 4.2))
y = np.arange(len(pr))
ax.errorbar(pr.beta_nonfinance, y - 0.15, xerr=1.96 * pr.se_nonfinance, fmt="o", color=st.PALETTE["other"], capsize=2, label="Non-finance")
ax.errorbar(pr.beta_finance, y + 0.15, xerr=1.96 * pr.se_finance, fmt="s", color=st.PALETTE["finance"], capsize=2, label="Finance")
ax.axvline(0, color="k", lw=0.7); ax.set_yticks(y)
ax.set_yticklabels([dict(order)[k].replace("$^{a}$", "").replace("$^{b}$", "") for k in plot_rows], fontsize=7.5)
ax.invert_yaxis(); ax.set_xlabel("Effect of log IPP capital per worker on the labor share (95% CI)"); ax.legend(fontsize=8)
fig.savefig(PF / "fig_robustness.pdf", bbox_inches="tight"); plt.show()

## Figure causal forest

In [ ]:
cs = ci.sort_values("cate").reset_index(drop=True)
NICE = {"5412OP": "Misc. professional services", "42": "Wholesale trade", "4A0": "Other retail",
        "482": "Rail transportation", "624": "Social assistance", "493": "Warehousing and storage"}
fig, ax = plt.subplots(figsize=(6.5, 6.0))
y = np.arange(len(cs))
cols = np.where(cs.finance == 1, st.PALETTE["finance"], st.PALETTE["other"])
ax.hlines(y, cs.ci_low, cs.ci_high, color=cols, lw=0.9, alpha=0.7)
ax.scatter(cs.cate, y, c=cols, s=np.where(cs.finance == 1, 40, 12), zorder=3)
ax.axvline(0, color="k", lw=0.7)
xmax = cs.ci_high.max()
for i, r in cs.iterrows():
    if r.finance == 1:
        ax.annotate(SHORT.get(r.code, r.code), (r.ci_high, i), xytext=(4, 0), textcoords="offset points",
                    va="center", fontsize=7, color=st.PALETTE["finance"], fontweight="bold")
    elif r["rank"] in (1, 2, 5) or r["rank"] >= len(cs) - 2:
        ax.annotate(f"{NICE.get(r.code, r['name'][:30])} ({r.code})", (r.ci_low, i), xytext=(-4, 0), textcoords="offset points",
                    va="center", ha="right", fontsize=6.5, color="#404040")
ax.set_yticks([]); ax.invert_yaxis()
ax.set_xlim(cs.ci_low.min() - 0.17, xmax + 0.09)
ax.set_xlabel("Estimated effect on the labor share (95% CI)")
ax.set_ylabel("63 industries, sorted by estimated effect")
fig.savefig(PF / "fig_cf_effects.pdf", bbox_inches="tight"); plt.show()

## Copy figures

In [ ]:
for src, dst in [("fig8b_h3_randomization.pdf", "fig_randomization.pdf"), ("fig16_cf_drivers.pdf", "fig_cf_drivers.pdf")]:
    if (FIGURES_DIR / src).exists():
        shutil.copy(FIGURES_DIR / src, PF / dst); print("copied", dst)
print(sorted(p.name for p in PT.glob("*.tex"))); print(sorted(p.name for p in PF.glob("*.pdf")))